In [ ]:
# Prior calculation

In [1]:
def get_pubchem_info_from_inchikey_debug(inchikey):
    print(f"\n🔍 Processing: {inchikey}")

    # Step 1: Get CID
    cid_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/inchikey/{inchikey}/cids/JSON"
    cid_response = requests.get(cid_url)
    print("CID URL:", cid_url)
    print("CID Status:", cid_response.status_code)
    print("CID Response:", cid_response.text)

    if cid_response.status_code != 200:
        return {"inchikey": inchikey, "xref_count": None, "pubmed_count": None, "patent_count": None}

    try:
        cids = cid_response.json()["IdentifierList"]["CID"]
        cid = cids[0]
    except Exception as e:
        print("❌ Failed to extract CID:", e)
        return {"inchikey": inchikey, "xref_count": None, "pubmed_count": None, "patent_count": None}

    # Step 2: Get Record
    record_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/record/JSON"
    record_response = requests.get(record_url)
    print("Record URL:", record_url)
    print("Record Status:", record_response.status_code)
    print("Record Response Snippet:", record_response.text[:300])  # limit to first 300 chars

    if record_response.status_code != 200:
        return {"inchikey": inchikey, "xref_count": None, "pubmed_count": None, "patent_count": None}

    try:
        record = record_response.json()["Record"]
        pubmed_count = record.get("ReferenceCount", None)
        patent_count = record.get("PatentCount", None)
        xref_count = len(record.get("RecordMetadata", {}).get("SourceList", []))
    except Exception as e:
        print("❌ Failed to parse Record:", e)
        pubmed_count = patent_count = xref_count = None

    return {
        "inchikey": inchikey,
        "xref_count": xref_count,
        "pubmed_count": pubmed_count,
        "patent_count": patent_count
    }


In [10]:

# Try it on full InChIKeys
inchikeys = [
    "BSYNRYMUTXBXSQ-UHFFFAOYSA-N",  # Caffeine
    "RZVAJINKPMORJF-UHFFFAOYSA-N",  # Aspirin
    "KHPQFUQHAYHUBC-UHFFFAOYSA-N",  # Acetaminophen
    "GZCGUPFRVQAUEE-UHFFFAOYSA-N"   # Glucose
]

results = []
for ik in inchikeys:
    print(f"Fetching for InChIKey: {ik}")
    data = get_pubchem_info_from_inchikey_full(ik)
    results.append(data)
    time.sleep(0.25)

df = pd.DataFrame(results)
print(df)


Fetching for InChIKey: BSYNRYMUTXBXSQ-UHFFFAOYSA-N
Fetching for InChIKey: RZVAJINKPMORJF-UHFFFAOYSA-N
Fetching for InChIKey: KHPQFUQHAYHUBC-UHFFFAOYSA-N
Fetching for InChIKey: GZCGUPFRVQAUEE-UHFFFAOYSA-N
                      inchikey xref_count pubmed_count patent_count
0  BSYNRYMUTXBXSQ-UHFFFAOYSA-N       None         None         None
1  RZVAJINKPMORJF-UHFFFAOYSA-N       None         None         None
2  KHPQFUQHAYHUBC-UHFFFAOYSA-N       None         None         None
3  GZCGUPFRVQAUEE-UHFFFAOYSA-N       None         None         None


In [3]:
import requests
import time
import numpy as np
import pandas as pd

# Recursive PubMed counter
def count_pubmed_references(sections):
    count = 0
    for section in sections:
        for info in section.get("Information", []):
            if "ReferenceNumber" in info:
                ref = info["ReferenceNumber"]
                if isinstance(ref, list):
                    count += len(ref)
                elif isinstance(ref, int):
                    count += 1
        if "Section" in section:
            count += count_pubmed_references(section["Section"])
    return count

# Recursive patent counter
def count_patents(sections):
    count = 0
    for section in sections:
        if section.get("TOCHeading") == "Patents":
            for info in section.get("Information", []):
                val = info.get("Value", {}).get("StringWithMarkup", [])
                if isinstance(val, list):
                    count += len(val)
        if "Section" in section:
            count += count_patents(section["Section"])
    return count

# Combine counts into a prior score
def compute_prior(pubmed_count, patent_count):
    pubmed = pubmed_count if pd.notnull(pubmed_count) else 0
    patent = patent_count if pd.notnull(patent_count) else 0

    score = 0.5 * min(1, np.log1p(pubmed) / 6) + \
            0.5 * min(1, np.log1p(patent) / 5)
    return round(min(1.0, score), 3)

# Main function to fetch data
def get_pubchem_counts_with_prior(inchikey):
    print(f"🔍 Looking up: {inchikey}")

    # Step 1: InChIKey → CID
    cid_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/inchikey/{inchikey}/cids/JSON"
    cid_resp = requests.get(cid_url)
    if cid_resp.status_code != 200:
        print(f"❌ CID lookup failed for {inchikey}")
        return {"inchikey": inchikey, "cid": None, "pubmed_count": None, "patent_count": None, "prior_score": 0.0}

    try:
        cid = cid_resp.json()["IdentifierList"]["CID"][0]
    except Exception as e:
        print(f"❌ Failed to extract CID: {e}")
        return {"inchikey": inchikey, "cid": None, "pubmed_count": None, "patent_count": None, "prior_score": 0.0}

    # Step 2: Fetch metadata from pug_view
    summary_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug_view/data/compound/{cid}/JSON"
    summary_resp = requests.get(summary_url)
    if summary_resp.status_code != 200:
        print(f"❌ Summary fetch failed for CID {cid}")
        return {"inchikey": inchikey, "cid": cid, "pubmed_count": None, "patent_count": None, "prior_score": 0.0}

    try:
        record = summary_resp.json()["Record"]
        sections = record["Section"]
        pubmed_count = count_pubmed_references(sections)
        patent_count = count_patents(sections)
        prior_score = compute_prior(pubmed_count, patent_count)
    except Exception as e:
        print(f"❌ Failed to parse summary record: {e}")
        pubmed_count = patent_count = 0
        prior_score = 0.0

    return {
        "inchikey": inchikey,
        "cid": cid,
        "pubmed_count": pubmed_count,
        "patent_count": patent_count,
        "prior_score": prior_score
    }

# 🔬 Example test run
inchikeys = [
    "BSYNRYMUTXBXSQ-UHFFFAOYSA-N",  # Caffeine
    "RZVAJINKPMORJF-UHFFFAOYSA-N",  # Aspirin
    "HUMNYLRZRPPJDN-UHFFFAOYSA-N",  # Theobromine
]

results = []
for ik in inchikeys:
    result = get_pubchem_counts_with_prior(ik)
    results.append(result)
    time.sleep(0.3)  # Respectful pause

df = pd.DataFrame(results)
print(df)


🔍 Looking up: BSYNRYMUTXBXSQ-UHFFFAOYSA-N
🔍 Looking up: RZVAJINKPMORJF-UHFFFAOYSA-N
🔍 Looking up: HUMNYLRZRPPJDN-UHFFFAOYSA-N
                      inchikey   cid  pubmed_count  patent_count  prior_score
0  BSYNRYMUTXBXSQ-UHFFFAOYSA-N  2244          1598            14        0.771
1  RZVAJINKPMORJF-UHFFFAOYSA-N  1983          2824            36        0.861
2  HUMNYLRZRPPJDN-UHFFFAOYSA-N   240           962             0        0.500
